# Phase 7 (Prior Case Retrieval) - Fine-tuning the Phase 1 models as retrievers

One notebook, several experiments run back to back (`RUNS` below), one run per GPU. Every run:

1. loads a Phase 1 checkpoint (Model 2 / 3 / 4 / 5) as a **bi-encoder**: a case is tokenized, a part of it is cut into
   chunks of `chunk_len` tokens (`section`, `n_chunks`), the backbone reads every chunk, the chunk's tokens are
   mean-pooled, the chunk vectors are averaged into one case vector, L2-normalised. Relevance = cosine similarity.
   Queries and candidates share the encoder (the queries are themselves in the candidate pools),
2. adapts it with the run's method (`lorma`, `ia3`, `partial`, `full`, `kd`),
3. trains on train.json with a contrastive loss (InfoNCE): each query against its relevant case, a BM25-mined hard
   negative and every other case in the batch (large batches via gradient caching: the batch's embeddings are computed
   without gradients, the loss gradient w.r.t. them is pushed back through the encoder in small pieces),
4. keeps the epoch with the best **Micro-F1@7 on dev** (= `evaluate_phase7.py`), ranking each query's own split pool,
5. saves the case embeddings of the dev / test pools (for fusion with BM25 and ensembles offline) and a submission:
   top 7 by cosine -> `<run>/submission.zip` (prediction.json at the root).

**Ranking rules used everywhere** (structure of the task, no labels): a query is never its own result; a candidate
decided after the query (header dates "DATE OF JUDGMENT") cannot be a prior case. The header field "CITATOR INFO"
(later cases that cited this one) is removed from every text: it would reveal the relevance labels.

**Inputs to add on Kaggle:** the HF secret `HF` and one dataset per Phase 1 model (as in Phases 4-6).
**Outputs:** `/kaggle/working/phase7_log.txt` and `/kaggle/working/phase7_results_bundle.zip` (logs, metrics,
embeddings, submissions) - download this one.

The hyperparameters below are starting values, not tuned ones.

In [ ]:
# ============================ CONFIG ============================
MODEL_DIRS = {          # folder (or any file inside it) of each Phase 1 checkpoint; None = auto-discover
    "m2": "/kaggle/input/datasets/sanskaars22/model-2-capstone",   # Model 2: A-3 decoder, RoPE + GQA          (a3_config.json + a3_model.pt)
    "m3": None,         # Model 3: A-2 GRU encoder-decoder           (gru_config.json + gru_model.pt, or Trainer *.safetensors)
    "m4": "/kaggle/input/datasets/sanskaars22/model-5",   # Model 4: modern decoder (this upload is legal-modern-dec-101m)
    "m5": "/kaggle/input/datasets/sanskaars22/model-4-prefix-lm-labelled-as-m5-in-notebook",   # Model 5: prefix-LM (legal-prefix-lm-101m)
}
DISCOVER_KEYWORDS = {"m2": ["model-2", "model_2", "model2", "dec-rope"], "m3": ["model-3", "model_3", "model3", "gru"],
                     "m4": ["model-4", "model_4", "model4"], "m5": ["model-5", "model_5", "model5", "prefix"]}

COMMON = dict(
    section="head",                # part of a case the model reads: "head" (first tokens), "tail", "middle", "head_tail",
                                   # "spread" (n_chunks evenly spaced chunks)
    chunk_len=512,                 # tokens per chunk (<= the model's max_seq_len)
    n_chunks=2,                    # chunks per case: the model reads n_chunks x chunk_len tokens of `section`
    attn="bidir",                  # decoders: "bidir" (no causal mask during fine-tuning) or "causal"
    temperature=0.05,              # InfoNCE: cosine / temperature
    queries_per_batch=16,          # each with 1 relevant case + `hard_negatives` BM25 negatives: ~48 cases per batch
    hard_negatives=1,
    hard_pool=30,                  # hard negatives drawn from the top-k BM25 candidates that are not relevant
    batch_tokens=4096,             # tokens per forward piece (gradient caching: memory does not grow with the batch)
    epochs=6,
    train_minutes=60,              # training time budget per run (Kaggle: 12 h per session): the learning-rate schedule
                                   # is fitted to the batches that fit in it (measured on the first 10), then training stops
    lr_backbone=3e-5,              # pretrained weights (full / partial / kd student)
    lr_adapter=3e-4,               # LoRMA parameters
    lr_ia3=3e-3,                   # IA3 scaling vectors
    weight_decay=0.01, warmup_frac=0.1, grad_clip=1.0,
    rank=8, alpha=16, adapter_targets="attn+mlp",   # LoRMA
    partial_trainable_frac=0.5,    # partial: train the top half of the layers
    eval_sections=[],              # after training: dev (and test) embeddings reading other parts ("name:K" = K chunks)
    kd_temperature=2.0, kd_alpha=0.5,
    student_layers=None,           # keep only the lowest N layers of the checkpoint (KD student; None = all layers)
    save_weights=False,            # trainable parameters of the best epoch (bf16) -> bundle
    seed=42,
)

# Techniques (spec 13.2.1 list): LoRMA (compulsory), IA3, partial fine-tuning, full fine-tuning, knowledge distillation.
# m3 (GRU) has no Linear layers inside its recurrent cells: it is fully fine-tuned ("full").
# A KD run is dict(name=..., model=..., method="kd", teacher=<a run in this session>, student_layers=N): it waits for its
# teacher, then trains the smaller student on the teacher's similarity distributions (T = kd_temperature) + the labels.
# Every run also scores the untrained Phase 1 model on dev first ("epoch 0": how Phase 1 representations transfer).
# Queue order = list order (2 GPUs, the KD student waits for m5_lorma). ~9 s per batch of 16 queries (52 batches/epoch,
# 6 epochs ~ 50-60 min) + ~10 min evaluation per run; the KD student (4 layers) ~30 min -> all eight ~5 h.
RUNS = [
    dict(name="m5_lorma",      model="m5", method="lorma", eval_sections=["head:1", "head:4", "spread:4"]),
    dict(name="m4_ia3",        model="m4", method="ia3"),
    dict(name="m5_kd_student", model="m5", method="kd", teacher="m5_lorma", student_layers=4),
    dict(name="m2_partial",    model="m2", method="partial"),
    dict(name="m3_full",       model="m3", method="full", lr_backbone=1e-4),       # RNN: a higher starting lr
    dict(name="m4_full",       model="m4", method="full"),
    # full fine-tuning vs LoRMA on m5, LoRMA on m2 (method comparisons; optional)
    dict(name="m5_full",       model="m5", method="full"),
    dict(name="m2_lorma",      model="m2", method="lorma"),
]
RUN_ONLY = ["m5_lorma"]   # runs behind the best submission; None = every run in RUNS
SMOKE = False              # True: 24 training queries, 12 dev queries, 1 epoch per selected run (checks loading, training,
                           # ranking and zipping); outputs go to /kaggle/working/smoke/

In [ ]:
import os, re, json, glob, math, time, random, zipfile, hashlib, gc, platform, threading
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("p7_work")
if SMOKE: WORK = os.path.join(WORK, "smoke")
os.makedirs(WORK, exist_ok=True)
LOG_PATH = os.path.join(WORK, "phase7_log.txt")
_log = open(LOG_PATH, "a", encoding="utf-8")

_log_lock = threading.Lock()
def log(*a):
    s = " ".join(str(x) for x in a)
    t = threading.current_thread().name
    if t.startswith("gpu"): s = "\n".join(f"[{t}] {l}" for l in s.split("\n"))   # parallel runs: which GPU worker
    with _log_lock:
        print(s, flush=True); _log.write(s + "\n"); _log.flush()

def section(t):
    log("\n" + "=" * 100); log(t); log("=" * 100)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# fp16 (+ loss scaling) on GPUs without bf16 tensor cores (T4), as the Phase 1 models were trained; bf16 on Ampere and newer
AMP_DTYPE = {"fp16": torch.float16, "bf16": torch.bfloat16}.get(os.environ.get("P7_AMP"), torch.bfloat16
            if DEVICE.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float16)
section(f"Phase 7 fine-tuning - {time.strftime('%Y-%m-%d %H:%M:%S')}")
log(f"torch {torch.__version__} | python {platform.python_version()} | device {DEVICE}"
    + (f" ({torch.cuda.get_device_name(0)}, {torch.cuda.device_count()} GPU)" if DEVICE.type == "cuda" else "")
    + f" | amp {AMP_DTYPE}")
log("COMMON =", json.dumps(COMMON))

In [ ]:
# ---- data: Phase_7_PCR from the official dataset ----
def load_phase7_files():
    local = os.environ.get("P7_LOCAL_DATA")                       # local testing only
    if local:
        return {f: os.path.join(r, f) for r, _, fs in os.walk(local) for f in fs}
    from huggingface_hub import HfApi, hf_hub_download
    token, why = os.environ.get("HF_TOKEN"), "HF_TOKEN env var not set"
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF")
        except Exception as e:
            why = f"Kaggle secret 'HF' not readable ({e!r})"
    assert token, (f"no Hugging Face token: {why}. In the Kaggle editor: Add-ons -> Secrets -> add/attach a secret "
                   "named exactly HF (value = your hf_... token) and make sure its checkbox is ON for this notebook.")
    try:
        log(f"HF login ok: {HfApi().whoami(token=token)['name']}")
    except Exception as e:
        raise RuntimeError(f"the HF token was found but is rejected by Hugging Face ({e!r}); "
                           "create a new read token at huggingface.co/settings/tokens") from None
    repo = "Exploration-Lab/CS-781-Capstone"
    files = [f for f in HfApi().list_repo_files(repo, repo_type="dataset", token=token) if f.startswith("Phase_7_PCR/")]
    return {os.path.basename(f): hf_hub_download(repo, f, repo_type="dataset", token=token) for f in files}

P7 = load_phase7_files()
read = lambda n: json.load(open(P7[n], encoding="utf-8"))

CITATOR = re.compile(r"CITATOR INFO\s*:.*?(?=\bACT:|\bHEADNOTE:|$)", re.S)
DATE_RE = re.compile(r"DATE OF JUDGMENT\s*:?\s*(\d{1,2})/(\d{1,2})/(\d{4})")
YEAR_RE = re.compile(r"\b(18[5-9]\d|19\d\d|20[0-2]\d)\b")

def clean_text(sents):
    """Sentences -> one text; the header field CITATOR INFO (the later cases citing this one) is removed."""
    head = CITATOR.sub(" ", " ".join(sents[:3]))
    return " ".join([head] + sents[3:])

def case_dates(text):
    """-> (lower bound, upper bound) of the decision date in years: the header date (both bounds) if there is one,
    else (latest year mentioned in the text, None): a case cannot mention a year after it was decided."""
    m = DATE_RE.search(text[:3000])
    if m:
        y = int(m[3]) + (int(m[2]) - 1) / 12 + (int(m[1]) - 1) / 372
        return y, y
    ys = [int(y) for y in YEAR_RE.findall(text)]
    return (max(ys) if ys else None), None

DOCS, SPLITS = {}, {}   # case id -> dict(text, lo, hi); split -> dict(queries, pool, rel)
for split, folder in (("train", "Train"), ("dev", "Dev"), ("test", "Test")):
    q, c = read(f"{split}.json"), read(f"{split}_candidates.json")
    for d, x in list(c.items()) + list(q.items()):
        if d in DOCS: continue                           # the same id always has the same text (checked offline)
        t = clean_text(x["text"]); lo, hi = case_dates(t)
        DOCS[d] = dict(text=t, lo=lo, hi=hi)
    pool = sorted(c)
    rel = None if split == "test" else {k: [y for y in v["relevant_candidates"] if y.strip()] for k, v in q.items()}
    queries = sorted(q) if rel is None else sorted(k for k in q if rel[k])   # train: 10 queries without relevant cases
    if rel is not None:
        assert all(y in c for r in rel.values() for y in r), f"{split}: relevant case outside the pool"
    SPLITS[split] = dict(queries=queries, pool=pool, rel=rel)
if SMOKE:
    for s, n in (("train", 24), ("dev", 12), ("test", 12)):
        S = SPLITS[s]; S["queries"] = S["queries"][:n]
        keep = set(S["queries"]) | {y for k in S["queries"] for y in (S["rel"] or {}).get(k, [])}
        S["pool"] = sorted(keep | set(S["pool"][:60]))
        if S["rel"]: S["rel"] = {k: S["rel"][k] for k in S["queries"]}

def valid_mask(split):
    """(queries, pool) bool: may this candidate be returned for this query? Not the query itself, and not a case
    decided after the query (candidate's earliest possible date > the query's header date)."""
    S = SPLITS[split]; pi = {d: j for j, d in enumerate(S["pool"])}
    M = np.ones((len(S["queries"]), len(S["pool"])), bool)
    lo = np.array([np.nan if DOCS[d]["lo"] is None else DOCS[d]["lo"] for d in S["pool"]])
    for i, q in enumerate(S["queries"]):
        if q in pi: M[i, pi[q]] = False
        hi = DOCS[q]["hi"]
        if hi is not None: M[i, lo > hi] = False         # nan > hi is False: unknown dates stay allowed
    return M

def f1_at_7(split, S_, mask=None):
    """Micro-F1@7 / precision / recall (= evaluate_phase7.py) of a (queries, pool) score matrix."""
    Sp = SPLITS[split]; X = np.where(valid_mask(split) if mask is None else mask, S_, -np.inf)
    top = np.argsort(-X, 1)[:, :7]
    tp = fp = fn = 0
    for i, q in enumerate(Sp["queries"]):
        p = {Sp["pool"][j] for j in top[i]}; g = set(Sp["rel"][q])
        tp += len(p & g); fp += len(p - g); fn += len(g - p)
    return dict(f1=2 * tp / max(1, 2 * tp + fp + fn), precision=tp / max(1, tp + fp), recall=tp / max(1, tp + fn))

def rankings(split, S_, k=7):
    Sp = SPLITS[split]; X = np.where(valid_mask(split), S_, -np.inf)
    top = np.argsort(-X, 1)[:, :k]
    return {q: [Sp["pool"][j] for j in top[i]] for i, q in enumerate(Sp["queries"])}

# ---- BM25 (lexical baseline, hard-negative mining) ----
def bm25(split, k1=1.2, b=0.75):
    """(queries, pool) BM25 scores; vocabulary and statistics of the split's pool, query terms weighted log(1 + tf)."""
    from sklearn.feature_extraction.text import CountVectorizer
    import scipy.sparse as sp
    S = SPLITS[split]
    cv = CountVectorizer(lowercase=True, stop_words="english", token_pattern=r"(?u)\b[a-z][a-z]+\b", min_df=2, dtype=np.float32)
    D = cv.fit_transform([DOCS[d]["text"] for d in S["pool"]]).tocsr()
    Q = cv.transform([DOCS[d]["text"] for d in S["queries"]]).tocsr()
    N, df = D.shape[0], np.bincount(D.indices, minlength=D.shape[1])
    idf = np.log(1 + (N - df + .5) / (df + .5)).astype(np.float32)
    dl = np.asarray(D.sum(1)).ravel(); C = D.tocoo()
    W = sp.csr_matrix((C.data * (k1 + 1) / (C.data + k1 * (1 - b + b * dl[C.row] / dl.mean())) * idf[C.col], (C.row, C.col)), D.shape)
    Q.data = np.log1p(Q.data)
    return (Q @ W.T).toarray()

section("DATA")
t0 = time.time()
BM25 = {s: bm25(s) for s in SPLITS}
for s, S in SPLITS.items():
    w = np.array([len(DOCS[d]["text"].split()) for d in S["pool"]]); M = valid_mask(s)
    log(f"{s:5s}: {len(S['queries'])} queries, pool {len(S['pool'])} cases (words median {int(np.median(w))}, p90 "
        f"{int(np.percentile(w, 90))}); header date known for {np.mean([DOCS[d]['hi'] is not None for d in S['pool']]):.0%} "
        f"of the pool; candidates allowed per query (median) {int(np.median(M.sum(1)))}")
    if S["rel"]:
        n = np.array([len(S["rel"][q]) for q in S["queries"]]); pi = {d: j for j, d in enumerate(S["pool"])}
        lost = sum(not M[i, pi[y]] for i, q in enumerate(S["queries"]) for y in S["rel"][q])
        log(f"       relevant per query: median {int(np.median(n))}, mean {n.mean():.2f}, max {n.max()}; relevant cases "
            f"excluded by the date rule: {lost} of {n.sum()}; BM25 Micro-F1@7 {f1_at_7(s, BM25[s])['f1']:.4f}")
log(f"BM25 for all splits in {time.time() - t0:.0f}s")

In [ ]:
# ---- Phase 1 byte-level BPE (same algorithm and ids as LegalBPETokenizer), with character offsets ----
WORD_RE = re.compile(r"\S+")
_SHIFT = 20
_EOW = (1 << _SHIFT) - 1

class OffsetBPE:
    def __init__(self, path):
        with open(path, encoding="utf-8") as f:
            d = json.load(f)
        self.itos = d["itos"]
        sym2id = {tuple(s): i for i, s in enumerate(self.itos) if isinstance(s, list)}
        to_id = lambda t: _EOW if tuple(t) == (256,) else sym2id[tuple(t)]
        self.ranks = {(to_id(l) << _SHIFT) | to_id(r): sym2id[tuple(l) + tuple(r)] for l, r in d["merges"]}
        self.nbytes = [sum(b != 256 for b in s) if isinstance(s, list) else 0 for s in self.itos]
        self.eow_id = len(self.itos)               # LegalBPETokenizer: a bare end-of-word symbol gets id len(itos)
        self.vocab_size = len(self.itos) + 1
        self.cache = {}

    def tokenize_word(self, word):
        out = self.cache.get(word)
        if out is not None: return out
        w = [b + 4 for b in word.encode("utf-8")] + [_EOW]
        ranks = self.ranks
        while len(w) > 1:
            best = None
            for x, y in zip(w, w[1:]):
                n = ranks.get((x << _SHIFT) | y)
                if n is not None and (best is None or n < best[0]): best = (n, x, y)
            if best is None: break
            n, a, b = best; i, res = 0, []
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b: res.append(n); i += 2
                else: res.append(w[i]); i += 1
            w = res
        self.cache[word] = w
        return w

    def encode(self, text):
        """-> ids, char_start, char_end (np arrays). A bare end-of-word token has an empty span at the word end."""
        ids, cs, ce = [], [], []
        for m in WORD_RE.finditer(text):
            word, base = m.group(), m.start()
            b2c = []
            for ci, ch in enumerate(word): b2c.extend([ci] * len(ch.encode("utf-8")))
            p = 0
            for t in self.tokenize_word(word):
                nb = 0 if t == _EOW else self.nbytes[t]
                ids.append(self.eow_id if t == _EOW else t)
                if nb == 0:
                    cs.append(base + len(word)); ce.append(base + len(word))
                else:
                    cs.append(base + b2c[p]); ce.append(base + b2c[p + nb - 1] + 1); p += nb
        return np.array(ids, np.int64), np.array(cs, np.int64), np.array(ce, np.int64)

# ---- From-scratch attention (no nn.MultiheadAttention / F.scaled_dot_product_attention), as in Phase 1 ----
# out = softmax(q k^T * scale, masked) v, computed in query blocks with the softmax statistics kept per row
# (log-sum-exp), so the full (T, S) score matrix is never stored. The backward pass is written by hand and
# recomputes each block's probabilities from the saved log-sum-exp:
#   P = exp(S - lse),  dV = P^T dO,  dP = dO V^T,  dS = P * (dP - rowsum(dO * O)),  dQ = dS K * scale,  dK = dS^T Q * scale
def _block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset):
    s = torch.matmul(q_blk, k[:, :, :kend].transpose(-2, -1)).float() * scale
    if attn_mask is not None:
        s = s.masked_fill(~attn_mask[..., i:j, :kend], float("-inf"))
    if is_causal:
        rows = torch.arange(i, j, device=s.device)[:, None] + offset
        s = s.masked_fill(torch.arange(kend, device=s.device)[None, :] > rows, float("-inf"))
    return s


class BlockwiseAttention(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, attn_mask, is_causal, scale, block):
        T, S = q.size(-2), k.size(-2)
        offset = S - T
        out = torch.empty_like(q)
        lse = torch.empty(q.shape[:-1], dtype=torch.float32, device=q.device)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S
            s = _block_scores(q[:, :, i:j], k, scale, attn_mask, is_causal, i, j, kend, offset)
            m = s.amax(-1, keepdim=True)
            p = torch.exp(s - m)
            l = p.sum(-1, keepdim=True)
            out[:, :, i:j] = torch.matmul((p / l).to(v.dtype), v[:, :, :kend])
            lse[:, :, i:j] = (m + l.log()).squeeze(-1)
        ctx.save_for_backward(q, k, v, out, lse, attn_mask)
        ctx.is_causal, ctx.scale, ctx.block = is_causal, scale, block
        return out

    @staticmethod
    def backward(ctx, d_out):
        q, k, v, out, lse, attn_mask = ctx.saved_tensors
        is_causal, scale, block = ctx.is_causal, ctx.scale, ctx.block
        T, S = q.size(-2), k.size(-2)
        offset = S - T
        d_out = d_out.contiguous()
        dq = torch.empty_like(q)
        dk = torch.zeros(k.shape, dtype=torch.float32, device=k.device)
        dv = torch.zeros(v.shape, dtype=torch.float32, device=v.device)
        delta = (d_out.float() * out.float()).sum(-1, keepdim=True)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S
            q_blk, do_blk = q[:, :, i:j], d_out[:, :, i:j]
            p = torch.exp(_block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset) - lse[:, :, i:j, None])
            dv[:, :, :kend] += torch.matmul(p.to(v.dtype).transpose(-2, -1), do_blk).float()
            dp = torch.matmul(do_blk, v[:, :, :kend].transpose(-2, -1)).float()
            ds = (p * (dp - delta[:, :, i:j])).to(q.dtype)
            dq[:, :, i:j] = torch.matmul(ds, k[:, :, :kend]) * scale
            dk[:, :, :kend] += torch.matmul(ds.transpose(-2, -1), q_blk).float() * scale
        return dq, dk.to(k.dtype), dv.to(v.dtype), None, None, None, None


def scratch_attention(q, k, v, attn_mask=None, is_causal=False, scale=None, block=128):
    """q (B, H, T, d), k/v (B, H_kv, S, d) with H_kv dividing H (GQA). attn_mask: bool (True = may attend)."""
    scale = q.size(-1) ** -0.5 if scale is None else scale
    B, H, T, d = q.shape
    n_rep = H // k.size(1)
    if n_rep > 1 and not is_causal:
        q = q.reshape(B, k.size(1), n_rep * T, d)
        if attn_mask is not None:
            attn_mask = attn_mask.repeat(1, 1, n_rep, 1)
        out = BlockwiseAttention.apply(q, k, v, attn_mask, False, scale, block)
        return out.reshape(B, H, T, d)
    if n_rep > 1:
        k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
    return BlockwiseAttention.apply(q, k, v, attn_mask, is_causal, scale, block)


def attend(q, k, v, key_keep, causal, scale=None):
    """causal or bidirectional attention; key_keep (B, T) bool marks real tokens (None = no padding)."""
    if key_keep is None:
        return scratch_attention(q, k, v, is_causal=causal, scale=scale)
    T = q.size(-2)
    allowed = key_keep[:, None, None, :].expand(-1, 1, T, -1)
    if causal:
        allowed = allowed & torch.ones(T, T, dtype=torch.bool, device=q.device).tril()
    allowed = allowed | torch.eye(T, dtype=torch.bool, device=q.device)   # padding rows see themselves (no NaN)
    return scratch_attention(q, k, v, attn_mask=allowed, scale=scale)


def rope_cache(T, d_head, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head))
    ang = torch.outer(torch.arange(T, dtype=torch.float32), inv_freq)
    ang = torch.cat([ang, ang], dim=-1)
    return ang.cos(), ang.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

In [ ]:
# ---- Phase 1 backbones as encoders. Parameter names match the Phase 1 checkpoints exactly. ----
class A3Attention(nn.Module):              # Model 2 (A-3 CausalSelfAttention: MHA/GQA, RoPE or sinusoidal)
    def __init__(self, c):
        super().__init__()
        self.n_heads, self.n_kv, self.d_head = c["n_heads"], c["n_kv_heads"], c["d_model"] // c["n_heads"]
        self.scale = c.get("attention_scale") or self.d_head ** -0.5
        self.q_proj = nn.Linear(c["d_model"], self.n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(c["d_model"], self.n_kv * self.d_head, bias=False)
        self.v_proj = nn.Linear(c["d_model"], self.n_kv * self.d_head, bias=False)
        self.out_proj = nn.Linear(self.n_heads * self.d_head, c["d_model"], bias=False)
        self.use_rope = c.get("pos_emb_type") == "rope"
        if self.use_rope:
            cos, sin = rope_cache(c["max_seq_len"], self.d_head, c.get("rope_base", 10000.0))
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, key_keep, causal):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv, self.d_head).transpose(1, 2)
        if self.use_rope:
            cos, sin = self.rope_cos[:T][None, None].to(q.dtype), self.rope_sin[:T][None, None].to(q.dtype)
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        out = attend(q, k, v, key_keep, causal, self.scale)
        return self.out_proj(out.transpose(1, 2).reshape(B, T, -1))


class A3Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.norm_first = c.get("norm_first", False)
        self.ln1 = nn.LayerNorm(c["d_model"])
        self.attn = A3Attention(c)
        self.ln2 = nn.LayerNorm(c["d_model"])
        self.ffn = nn.Module()
        self.ffn.net = nn.Sequential(nn.Linear(c["d_model"], c["d_ff"]), nn.GELU(), nn.Linear(c["d_ff"], c["d_model"]), nn.Dropout(0.0))

    def forward(self, x, key_keep, causal):
        if self.norm_first:
            x = x + self.attn(self.ln1(x), key_keep, causal)
            return x + self.ffn.net(self.ln2(x))
        x = self.ln1(x + self.attn(x, key_keep, causal))
        return self.ln2(x + self.ffn.net(x))


class A3Backbone(nn.Module):
    kind = "a3"
    def __init__(self, c):
        super().__init__()
        assert c.get("attention_type", "standard") == "standard", "only the standard (MHA/GQA) A-3 attention is supported"
        self.c, self.hidden_size = c, c["d_model"]
        self.tok_emb = nn.Embedding(c["vocab_size"], c["d_model"])
        if c.get("pos_emb_type") == "sinusoidal":
            pos = torch.arange(c["max_seq_len"], dtype=torch.float32)[:, None]
            div = torch.exp(torch.arange(0, c["d_model"], 2, dtype=torch.float32) * (-math.log(10000.0) / c["d_model"]))
            pe = torch.zeros(c["max_seq_len"], c["d_model"]); pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
            self.register_buffer("pos_emb", pe, persistent=False)
        self.blocks = nn.ModuleList([A3Block(c) for _ in range(c["n_layers"])])
        self.ln_f = nn.LayerNorm(c["d_model"])

    def forward(self, ids, key_keep=None, causal=False):
        x = self.tok_emb(ids)
        if hasattr(self, "pos_emb"): x = x + self.pos_emb[:ids.size(1)][None]
        for b in self.blocks: x = b(x, key_keep, causal)
        return self.ln_f(x)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).to(x.dtype) * self.weight


class ModernAttention(A3Attention):         # Models 4/5 GQAttention (always RoPE), same projections
    def __init__(self, c):
        super().__init__({**c, "pos_emb_type": "rope", "attention_scale": None})


class ModernBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.attn_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))
        self.attn = ModernAttention(c)
        self.mlp_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))
        self.mlp = nn.Module()
        self.mlp.gate = nn.Linear(c["d_model"], c["d_ff"], bias=False)
        self.mlp.up = nn.Linear(c["d_model"], c["d_ff"], bias=False)
        self.mlp.down = nn.Linear(c["d_ff"], c["d_model"], bias=False)

    def forward(self, x, key_keep, causal):
        x = x + self.attn(self.attn_norm(x), key_keep, causal)
        h = self.mlp_norm(x)
        return x + self.mlp.down(F.silu(self.mlp.gate(h)) * self.mlp.up(h))


class ModernBackbone(nn.Module):
    kind = "modern"
    def __init__(self, c):
        super().__init__()
        self.c, self.hidden_size = c, c["d_model"]
        self.tok_emb = nn.Embedding(c["vocab_size"], c["d_model"])
        self.blocks = nn.ModuleList([ModernBlock(c) for _ in range(c["n_layers"])])
        self.final_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))

    def forward(self, ids, key_keep=None, causal=False):
        x = self.tok_emb(ids)
        for b in self.blocks: x = b(x, key_keep, causal)
        return self.final_norm(x)


class GRUBackbone(nn.Module):               # Model 3: the bidirectional GRU encoder of the A-2 seq2seq model
    kind = "gru"
    def __init__(self, c):
        super().__init__()
        self.c, self.hidden_size = c, 2 * c["hidden"]
        self.embedding = nn.Embedding(c["vocab_size"], c["emb_dim"])
        self.enc_rnn = nn.GRU(c["emb_dim"], c["hidden"], num_layers=c["enc_layers"], batch_first=True, bidirectional=True)

    def forward(self, ids, key_keep=None, causal=False):
        self.enc_rnn.flatten_parameters()
        x = self.embedding(ids)
        if key_keep is None:
            return self.enc_rnn(x)[0]
        lengths = key_keep.sum(1).clamp(min=1).cpu()
        out, _ = self.enc_rnn(pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False))
        return pad_packed_sequence(out, batch_first=True, total_length=ids.size(1))[0]

In [ ]:
# ---- finding and loading the checkpoints ----
def _ckpt_kind(folder):
    names = os.listdir(folder)
    if "a3_config.json" in names: return "a3"
    if "modern_config.json" in names: return "modern"
    if "gru_config.json" in names: return "gru"
    st = [n for n in names if n.endswith(".safetensors")]
    if st:
        from safetensors import safe_open
        with safe_open(os.path.join(folder, st[0]), "pt") as f:
            keys = list(f.keys())
        if any("enc_rnn" in k for k in keys): return "gru"
        if any("attn_norm" in k for k in keys): return "modern"
        if any(".ln1." in k for k in keys): return "a3"
    return None

WANT_KIND = {"m2": "a3", "m3": "gru", "m4": "modern", "m5": "modern"}

def find_checkpoint(key):
    p = MODEL_DIRS.get(key)
    if p:
        folder = p if os.path.isdir(p) else os.path.dirname(p)
        if _ckpt_kind(folder) is None:          # a dataset root: look one level deeper
            subs = [os.path.dirname(f) for f in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                    if f.endswith(("_config.json", ".safetensors"))]
            subs = [s for s in dict.fromkeys(subs) if "checkpoint-" not in s and _ckpt_kind(s)]
            assert subs, f"no checkpoint found under {p}"
            folder = subs[0]
        return folder
    cands = []
    for f in glob.glob("/kaggle/input/**/*", recursive=True):
        if f.endswith(("_config.json", ".safetensors")):
            folder = os.path.dirname(f)
            if "checkpoint-" in folder: continue
            if _ckpt_kind(folder) == WANT_KIND[key]: cands.append(folder)
    cands = list(dict.fromkeys(cands))
    hit = [c for c in cands if any(w in c.lower() for w in DISCOVER_KEYWORDS[key])]
    shared = sum(k == WANT_KIND[key] for k in WANT_KIND.values()) > 1   # m4/m5: same format, never guess
    assert hit or (len(cands) == 1 and not shared), \
        f"{key}: cannot pick a checkpoint automatically from {cands}; set MODEL_DIRS['{key}']"
    return (hit or cands)[0]

def _strip(sd):
    out = {}
    for k, v in sd.items():
        for pre in ("module.", "lm.", "model."):
            if k.startswith(pre): k = k[len(pre):]
        out[k] = v
    return out

def load_backbone(key, n_layers=None):
    """Phase 1 checkpoint -> (backbone, info). n_layers < full depth keeps only the lowest layers (KD student)."""
    folder = find_checkpoint(key)
    kind = _ckpt_kind(folder)
    names = os.listdir(folder)
    pt = [n for n in names if n.endswith("_model.pt")]
    if pt:
        sd = torch.load(os.path.join(folder, pt[0]), map_location="cpu", weights_only=True)
    else:
        from safetensors.torch import load_file
        sd = load_file(os.path.join(folder, [n for n in names if n.endswith(".safetensors")][0]))
    sd = _strip(sd)
    cfg_name = {"a3": "a3_config.json", "modern": "modern_config.json", "gru": "gru_config.json"}[kind]
    if cfg_name in names:
        c = json.load(open(os.path.join(folder, cfg_name)))
    elif kind == "gru":                          # Trainer checkpoint without config: read the sizes off the weights
        c = dict(vocab_size=sd["embedding.weight"].shape[0], emb_dim=sd["embedding.weight"].shape[1],
                 hidden=sd["enc_rnn.weight_hh_l0"].shape[1],
                 enc_layers=len([k for k in sd if re.fullmatch(r"enc_rnn\.weight_ih_l\d+", k)]))
    else:
        raise FileNotFoundError(f"{folder}: {cfg_name} is needed to know the number of heads")
    full_layers = c["enc_layers"] if kind == "gru" else c["n_layers"]
    if n_layers:
        assert n_layers <= full_layers
        if kind == "gru":
            c = {**c, "enc_layers": n_layers}
            sd = {k: v for k, v in sd.items() if not re.search(r"_l(\d+)", k) or int(re.search(r"_l(\d+)", k).group(1)) < n_layers}
        else:
            c = {**c, "n_layers": n_layers}
            sd = {k: v for k, v in sd.items() if not k.startswith("blocks.") or int(k.split(".")[1]) < n_layers}
    model = {"a3": A3Backbone, "modern": ModernBackbone, "gru": GRUBackbone}[kind](c)
    own = model.state_dict()
    used = {k: v for k, v in sd.items() if k in own}
    missing = [k for k in own if k not in used]
    assert not missing, f"weights missing from the checkpoint: {missing[:5]}"
    for k, v in used.items():
        assert v.shape == own[k].shape, f"{k}: checkpoint {tuple(v.shape)} vs model {tuple(own[k].shape)}"
    model.load_state_dict(used)
    tok = [os.path.join(folder, n) for n in names if n.startswith("bpe_tokenizer") and n.endswith(".json")]
    for where in (os.path.join(os.path.dirname(folder), "**"), "/kaggle/input/**"):     # next to the weights, then anywhere
        tok = tok or sorted(glob.glob(os.path.join(where, "bpe_tokenizer*.json"), recursive=True))
    assert tok, f"no bpe_tokenizer*.json found for {key}: add the Phase 1 tokenizer as an input"
    info = dict(folder=folder, kind=kind, config=c, full_layers=full_layers, n_layers=n_layers or full_layers,
                unused_keys=len(sd) - len(used), tokenizer=tok[0], prefix_lm=c.get("prefix_lm"))
    return model, info

In [ ]:
# ---- adaptation methods ----
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2021): h = W0 x + (alpha / r) B A x, A (r x in) random, B (out x r) = 0."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + (x @ self.A.t().to(x.dtype)) @ self.B.t().to(x.dtype) * self.scale


class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany, Patel & Modi 2025): multiplicative update h = (I + (alpha / r) B A) W0 x, with A (r x out) random,
    B (out x r) = 0 so training starts at W0. Computed as z = W0 x, then z + (alpha / r) B (A z): never forms the d x d matrix."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + (z @ self.A.t().to(z.dtype)) @ self.B.t().to(z.dtype) * self.scale
        return z if self.base.bias is None else z + self.base.bias



class RSLoRALinear(LoRALinear):
    """rsLoRA (Kalajdzievski 2023): LoRA with the update scaled by alpha / sqrt(r) instead of alpha / r, so the update
    does not shrink as the rank grows."""
    def __init__(self, base, r, alpha):
        super().__init__(base, r, alpha)
        self.scale = alpha / math.sqrt(r)


def adapter_target_names(backbone, targets):
    names = []
    for n, m in backbone.named_modules():
        if not isinstance(m, nn.Linear): continue
        is_attn = ".attn." in n
        is_mlp = ".ffn." in n or ".mlp." in n
        if (is_attn and "attn" in targets) or (is_mlp and "mlp" in targets): names.append(n)
    return names

def inject(backbone, cls, targets, r, alpha):
    names = adapter_target_names(backbone, targets)
    for n in names:
        parent, child = backbone.get_submodule(n.rsplit(".", 1)[0]), n.rsplit(".", 1)[1]
        setattr(parent, child, cls(getattr(parent, child), r, alpha))
    return names

In [ ]:
# ---- cases -> token ids (Phase 1 tokenizer), parts of a case -> chunks, BM25 hard negatives ----
_TOKENS, _TOK_LOCK = {}, threading.Lock()
def get_tokens(tok_path):
    """Every case as one int32 array of token ids, cached per tokenizer file content (computed once per session)."""
    with _TOK_LOCK:
        h = hashlib.md5(open(tok_path, "rb").read()).hexdigest()
        if h not in _TOKENS:
            t0 = time.time(); bpe = OffsetBPE(tok_path); cache = {}
            def enc(text):
                out = []
                for w in text.split():                   # = WORD_RE words
                    t = cache.get(w)
                    if t is None:
                        t = cache[w] = [bpe.eow_id if i == _EOW else i for i in bpe.tokenize_word(w)]
                    out.extend(t)
                return np.array(out or [bpe.eow_id], np.int32)
            _TOKENS[h] = {d: enc(x["text"]) for d, x in DOCS.items()}
            bpe.cache.clear()
            n = np.array([len(v) for v in _TOKENS[h].values()])
            log(f"tokenizer {tok_path} (md5 {h[:8]}): {len(DOCS)} cases in {time.time() - t0:.0f}s; tokens/case median "
                f"{int(np.median(n))}, p25 {int(np.percentile(n, 25))}, p75 {int(np.percentile(n, 75))}, max {n.max()}; "
                f"{100 * np.mean(n <= 1024):.1f}% <= 1024, {100 * np.mean(n <= 4096):.1f}% <= 4096")
        return _TOKENS[h]

def parse_section(spec, K):
    """"head" -> ("head", K); "head:4" -> ("head", 4)."""
    name, _, k = spec.partition(":")
    assert name in ("tail", "head", "middle", "head_tail", "spread"), f"unknown section {spec}"
    return name, int(k) if k else K

def select_chunks(t, name, K, L):
    """The part `name` of a case (K x L tokens), as a list of <= K chunks of <= L tokens (in document order)."""
    n, B = len(t), K * L
    if name == "spread":                                 # K chunks of L tokens evenly spaced over the case
        starts = sorted(set(np.linspace(0, max(0, n - L), K).round().astype(int).tolist()))
        return [t[s:s + L] for s in starts]
    if n <= B: region = t
    elif name == "tail": region = t[n - B:]
    elif name == "head": region = t[:B]
    elif name == "middle": region = t[(n - B) // 2:(n - B) // 2 + B]
    else: region = np.concatenate([t[:B // 2], t[n - (B - B // 2):]])     # head_tail
    return [region[s:s + L] for s in range(0, len(region), L)]

def case_chunks(toks, ids, spec, cfg):
    name, K = parse_section(spec, cfg["n_chunks"])
    return {d: select_chunks(toks[d], name, K, cfg["chunk_len"]) for d in ids}

def case_batches(ids, chunks, budget):
    """Consecutive groups of cases with (number of chunks) x (longest chunk) <= budget tokens (>= 1 per batch)."""
    out, cur, nc, mx = [], [], 0, 0
    for d in ids:
        c, L = len(chunks[d]), max(len(x) for x in chunks[d])
        if cur and (nc + c) * max(mx, L) > budget:
            out.append(cur); cur, nc, mx = [], 0, 0
        cur.append(d); nc += c; mx = max(mx, L)
    if cur: out.append(cur)
    return out

def make_batch(ids, chunks):
    """-> token ids (C, T) of all chunks, key_keep (None if no padding), chunk_case (C,) = index of the case in `ids`."""
    rows = [(b, c) for b, d in enumerate(ids) for c in chunks[d]]
    T = max(len(c) for _, c in rows)
    x = torch.zeros(len(rows), T, dtype=torch.long); keep = torch.zeros(len(rows), T, dtype=torch.bool)
    for r, (_, c) in enumerate(rows):
        x[r, :len(c)] = torch.from_numpy(c.astype(np.int64)); keep[r, :len(c)] = True
    return x, (None if bool(keep.all()) else keep), torch.tensor([b for b, _ in rows])

def hard_negative_lists(split, k):
    """query -> the k best BM25 candidates that are allowed (valid_mask) and not relevant (same for every run)."""
    S = SPLITS[split]; X = np.where(valid_mask(split), BM25[split], -np.inf); order = np.argsort(-X, 1)[:, :k + 64]
    out = {}
    for i, q in enumerate(S["queries"]):
        r = set(S["rel"][q])
        out[q] = [S["pool"][j] for j in order[i] if np.isfinite(X[i, j]) and S["pool"][j] not in r][:k]
    return out

HARD = hard_negative_lists("train", COMMON["hard_pool"])

In [ ]:
# ---- bi-encoder: chunk token mean-pooling -> mean over the case's chunks -> L2-normalised case vector ----
class BiEncoder(nn.Module):
    def __init__(self, backbone, causal):
        super().__init__()
        self.backbone, self.causal = backbone, causal

    def forward(self, ids, key_keep, chunk_case, n_cases):
        """-> (n_cases, d) float32, unit length."""
        h = self.backbone(ids, key_keep, self.causal).float()                # (C, T, d)
        if key_keep is None: v = h.mean(1)
        else:
            m = key_keep.float()[..., None]
            v = (h * m).sum(1) / m.sum(1).clamp(min=1)                       # (C, d): one vector per chunk
        z = torch.zeros(n_cases, v.size(1), device=v.device)
        n = torch.zeros(n_cases, device=v.device).index_add_(0, chunk_case, torch.ones(len(v), device=v.device))
        case = z.index_add_(0, chunk_case, v) / n[:, None]
        return F.normalize(case, dim=-1)


class IA3Linear(nn.Module):
    """IA3 (Liu et al. 2022): a learned rescaling vector l, initialised to 1 (training starts at the Phase 1 model).
    "out": h = l * (W x)  (keys, values);  "in": h = W (l * x)  (the feed-forward output layer: scales the intermediate
    activation)."""
    def __init__(self, base, where):
        super().__init__()
        self.base, self.where = base, where
        self.l = nn.Parameter(torch.ones(base.out_features if where == "out" else base.in_features))

    def forward(self, x):
        if self.where == "out":
            y = self.base(x)
            return y * self.l.to(y.dtype)                # in the output's dtype (autocast: keys / values stay fp16)
        return self.base(x * self.l.to(x.dtype))


def inject_ia3(bb):
    names = []
    for n, m in list(bb.named_modules()):
        if not isinstance(m, nn.Linear): continue
        where = "out" if n.endswith(("attn.k_proj", "attn.v_proj")) else "in" if n.endswith(("mlp.down", "ffn.net.2")) else None
        if where is None: continue
        parent, child = bb.get_submodule(n.rsplit(".", 1)[0]), n.rsplit(".", 1)[1]
        setattr(parent, child, IA3Linear(getattr(parent, child), where)); names.append(n)
    return names


def setup_method(model, cfg):
    """Freezes / wraps parameters for the run's method. Returns a list of (param group name, params, lr)."""
    bb, method = model.backbone, cfg["method"]
    for p in bb.parameters(): p.requires_grad = method in ("full", "kd", "partial")
    adapters, lr_ad = [], cfg["lr_adapter"]
    if method == "partial":                          # train the top layers only
        (bb.embedding if bb.kind == "gru" else bb.tok_emb).requires_grad_(False)
        if bb.kind == "gru":
            L = bb.c["enc_layers"]; keep_from = L - max(1, round(L * cfg["partial_trainable_frac"]))
            for n, p in bb.enc_rnn.named_parameters():
                if int(re.search(r"_l(\d+)", n).group(1)) < keep_from: p.requires_grad = False
        else:
            L = len(bb.blocks); keep_from = L - max(1, round(L * cfg["partial_trainable_frac"]))
            for b in bb.blocks[:keep_from]: b.requires_grad_(False)
        log(f"partial: layers {keep_from}..{L - 1} trainable")
    elif method in ("lora", "lorma"):
        assert bb.kind != "gru", f"{method} is implemented for Linear layers; the GRU encoder has none"
        names = inject(bb, {"lora": LoRALinear, "lorma": LoRMALinear}[method], cfg["adapter_targets"], cfg["rank"], cfg["alpha"])
        adapters = [p for n, p in bb.named_parameters() if n.endswith((".A", ".B"))]
        log(f"{method}: rank {cfg['rank']}, alpha {cfg['alpha']}, {len(names)} adapted Linear layers, e.g. {names[:4]}")
    elif method == "ia3":
        assert bb.kind != "gru", "ia3 is implemented for Linear layers; the GRU encoder has none"
        names = inject_ia3(bb)
        adapters, lr_ad = [p for n, p in bb.named_parameters() if n.endswith(".l")], cfg["lr_ia3"]
        log(f"ia3: {len(names)} rescaled Linear layers, e.g. {names[:3]}")
    else:
        assert method in ("full", "kd"), f"unknown method {method}"
    for p in adapters: p.requires_grad = True
    ad_ids = {id(p) for p in adapters}
    backbone_params = [p for p in bb.parameters() if p.requires_grad and id(p) not in ad_ids]
    return [("adapter", adapters, lr_ad), ("backbone", backbone_params, cfg["lr_backbone"])]

In [ ]:
# ---- embeddings and evaluation ----
@torch.no_grad()
def encode(model, chunks, ids, cfg):
    """-> (len(ids), d) float32 CPU tensor, rows in the order of `ids`."""
    model.eval(); out = {}
    order = sorted(ids, key=lambda d: (len(chunks[d]), max(len(c) for c in chunks[d])))   # similar sizes: less padding
    for grp in case_batches(order, chunks, 2 * cfg["batch_tokens"]):
        x, keep, cc = make_batch(grp, chunks)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            e = model(x.to(DEVICE), None if keep is None else keep.to(DEVICE), cc.to(DEVICE), len(grp)).float().cpu()
        for b, d in enumerate(grp): out[d] = e[b]
    return torch.stack([out[d] for d in ids])

def split_cases(split):
    S = SPLITS[split]
    return list(dict.fromkeys(S["queries"] + S["pool"]))

def split_scores(split, ids, E):
    """Cosine scores (queries, pool) from the embeddings E of `ids` (= split_cases(split))."""
    S = SPLITS[split]; ix = {d: i for i, d in enumerate(ids)}
    return (E[[ix[q] for q in S["queries"]]] @ E[[ix[d] for d in S["pool"]]].T).numpy()

def recall_at(split, S_, k):
    Sp = SPLITS[split]; X = np.where(valid_mask(split), S_, -np.inf); top = np.argsort(-X, 1)[:, :k]
    hit = sum(len({Sp["pool"][j] for j in top[i]} & set(Sp["rel"][q])) for i, q in enumerate(Sp["queries"]))
    return hit / sum(len(Sp["rel"][q]) for q in Sp["queries"])

def evaluate(model, toks, spec, cfg, split="dev"):
    ids = split_cases(split)
    E = encode(model, case_chunks(toks, ids, spec, cfg), ids, cfg)
    S_ = split_scores(split, ids, E)
    return E, S_, (dict(f1_at_7(split, S_), recall_at_100=recall_at(split, S_, 100)) if SPLITS[split]["rel"] else None)

fmt = lambda m: f"Micro-F1@7 {m['f1']:.4f} (P {m['precision']:.4f}, R {m['recall']:.4f}, R@100 {m['recall_at_100']:.4f})"

In [ ]:
# ---- one experiment ----
RESULTS, TEACHER_EMB = [], {}

def count_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))

def contrastive_loss(E, qi, ti, excl, tau):
    """InfoNCE over the batch's cases: row b = query qi[b], target ti[b]; excl (B, n) bool = not a negative (the query
    itself, its other relevant cases)."""
    S = (E[qi] @ E.T / tau).masked_fill(excl, -1e4)
    return F.cross_entropy(S, ti), S

def run_experiment(run):
    cfg = {**COMMON, **run}
    if SMOKE: cfg.update(epochs=1)
    name = cfg["name"]
    out_dir = os.path.join(WORK, name); os.makedirs(out_dir, exist_ok=True)
    section(f"RUN {name}: model {cfg['model']}, method {cfg['method']}, section {cfg['section']}, "
            f"{cfg['n_chunks']} x {cfg['chunk_len']} tokens")
    log("config:", json.dumps({k: v for k, v in cfg.items() if k not in COMMON or COMMON[k] != v}))
    set_seed(cfg["seed"])
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()

    backbone, info = load_backbone(cfg["model"], cfg["student_layers"])
    log(f"checkpoint {info['folder']} (kind {info['kind']}, layers {info['n_layers']}/{info['full_layers']}, "
        f"prefix_lm={info['prefix_lm']}, unused checkpoint tensors {info['unused_keys']})")
    if cfg["model"] in ("m4", "m5"):                     # same file format: make sure it is the right checkpoint
        assert bool(info["prefix_lm"]) == (cfg["model"] == "m5"), (
            f"{cfg['model']} loaded a checkpoint with prefix_lm={info['prefix_lm']} - wrong MODEL_DIRS['{cfg['model']}']?")
    if info["kind"] != "gru":
        assert cfg["chunk_len"] <= info["config"]["max_seq_len"], "chunk_len longer than the model's context"
    toks = get_tokens(info["tokenizer"])
    model = BiEncoder(backbone, causal=(cfg["attn"] == "causal" and info["kind"] != "gru"))
    groups = setup_method(model, cfg)
    model.to(DEVICE)
    n_total, n_train = count_params(model), count_params(model, True)
    log(f"parameters: total {n_total / 1e6:.2f}M, trainable {n_train / 1e6:.3f}M ({100 * n_train / n_total:.2f}%)")

    TR = SPLITS["train"]; rel_tr = {q: set(r) for q, r in TR["rel"].items()}
    tr_ch = case_chunks(toks, split_cases("train"), cfg["section"], cfg)
    teacher = None
    if cfg["method"] == "kd":
        teacher = TEACHER_EMB[cfg["teacher"]]
        log(f"kd: teacher {cfg['teacher']}, temperature {cfg['kd_temperature']}, alpha {cfg['kd_alpha']} "
            f"(loss = (1 - alpha) InfoNCE + alpha T^2 KL(teacher || student) over the batch's similarity distributions)")

    nq, B = len(TR["queries"]), cfg["queries_per_batch"]
    n_batches = math.ceil(nq / B)
    per_case = cfg["n_chunks"] * cfg["chunk_len"]
    log(f"train: {nq} queries, {sum(len(r) for r in rel_tr.values())} relevant pairs; {n_batches} batches/epoch of {B} "
        f"queries x (query + 1 relevant + {cfg['hard_negatives']} hard negatives) <= {B * (2 + cfg['hard_negatives']) * per_case / 1e3:.0f}k tokens")
    pg = [dict(params=[p for p in ps if p.ndim >= 2], lr=lr, weight_decay=cfg["weight_decay"]) for _, ps, lr in groups] + \
         [dict(params=[p for p in ps if p.ndim < 2], lr=lr, weight_decay=0.0) for _, ps, lr in groups]
    opt = torch.optim.AdamW([g for g in pg if g["params"]])
    plan = dict(steps=n_batches * cfg["epochs"], warm=max(1, int(cfg["warmup_frac"] * n_batches * cfg["epochs"])))
    lr_at = lambda s: ((s + 1) / plan["warm"] if s < plan["warm"]                # warmup, linear decay to 0 at the planned end
                       else max(0.0, (plan["steps"] - s) / max(1, plan["steps"] - plan["warm"])))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=(amp and AMP_DTYPE == torch.float16))
    params = [p for g in opt.param_groups for p in g["params"]]

    # epoch 0 = the Phase 1 model before any training
    t0 = time.time()
    _, _, vm = evaluate(model, toks, cfg["section"], cfg)
    history = [dict(epoch=0, train_loss=None, **{f"val_{k}": v for k, v in vm.items()}, seconds=time.time() - t0)]
    log(f"  epoch  0 (Phase 1 model, no training): dev {fmt(vm)} ({time.time() - t0:.0f}s)")
    best, best_state = vm["f1"], {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
    init_state = best_state
    def moved():                                         # relative change of the trainable weights since epoch 0
        num = sum(float((v.detach().float().cpu() - init_state[k].float()).pow(2).sum()) for k, v in model.named_parameters() if v.requires_grad)
        return math.sqrt(num / max(1e-12, sum(float(v.float().pow(2).sum()) for v in init_state.values())))

    rng = random.Random(cfg["seed"])                     # own generator: parallel runs do not disturb each other
    t_train, done, stop, budget, tok_seen = time.time(), 0, False, 60 * cfg["train_minutes"], 0
    for ep in range(cfg["epochs"]):
        model.train(); t0 = time.time()
        order = list(TR["queries"]); rng.shuffle(order)
        tot = kdt = 0.0; nb = 0
        for bi in range(n_batches):
            if done == 10 and not SMOKE:                 # fit the schedule to the batches that fit in the time budget
                fit = int(budget / ((time.time() - t_train) / 10))
                if fit < plan["steps"]:
                    plan.update(steps=fit, warm=max(1, int(cfg["warmup_frac"] * fit)))
                log(f"    {(time.time() - t_train) / 10:.1f} s/batch -> {plan['steps']} batches in the "
                    f"{cfg['train_minutes']} min budget ({plan['steps'] / n_batches:.2f} epochs)")
            if done >= plan["steps"] or time.time() - t_train > budget:
                stop = True; break
            done += 1
            qs = order[bi * B:(bi + 1) * B]
            pos = [rng.choice(TR["rel"][q]) for q in qs]
            negs = [d for q in qs for d in rng.sample(HARD[q], min(cfg["hard_negatives"], len(HARD[q])))]
            cases = list(dict.fromkeys(qs + pos + negs)); ix = {d: i for i, d in enumerate(cases)}
            qi = torch.tensor([ix[q] for q in qs], device=DEVICE); ti = torch.tensor([ix[p] for p in pos], device=DEVICE)
            excl = torch.zeros(len(qs), len(cases), dtype=torch.bool)
            for b, q in enumerate(qs):
                excl[b, ix[q]] = True
                for d in rel_tr[q]:
                    if d in ix and d != pos[b]: excl[b, ix[d]] = True
            excl = excl.to(DEVICE)
            pieces = case_batches(cases, tr_ch, cfg["batch_tokens"])
            # gradient caching: 1) embeddings without gradients, 2) loss and its gradient w.r.t. the embeddings,
            # 3) each piece again with gradients, back-propagating (embedding . its gradient)
            with torch.no_grad():
                E = []
                for p in pieces:
                    x, keep, cc = make_batch(p, tr_ch)
                    with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=amp):
                        E.append(model(x.to(DEVICE), None if keep is None else keep.to(DEVICE), cc.to(DEVICE), len(p)).float())
                    tok_seen += x.numel()
            E = torch.cat(E).requires_grad_(True)
            loss, S = contrastive_loss(E, qi, ti, excl, cfg["temperature"])
            if teacher is not None:                      # the teacher's similarity distribution over the same cases
                TE = torch.stack([teacher[d] for d in cases]).float().to(DEVICE)
                St = (TE[qi] @ TE.T / cfg["temperature"]).masked_fill(excl, -1e4)
                T_ = cfg["kd_temperature"]
                kd = F.kl_div(F.log_softmax(S / T_, -1), F.log_softmax(St / T_, -1), log_target=True,
                              reduction="batchmean") * T_ * T_
                kdt += kd.item()
                loss = (1 - cfg["kd_alpha"]) * loss + cfg["kd_alpha"] * kd
            loss.backward()
            G = E.grad.detach(); k0 = 0
            for p in pieces:
                x, keep, cc = make_batch(p, tr_ch)
                with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=amp):
                    e = model(x.to(DEVICE), None if keep is None else keep.to(DEVICE), cc.to(DEVICE), len(p)).float()
                scaler.scale((e * G[k0:k0 + len(p)]).sum()).backward(); k0 += len(p)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, cfg["grad_clip"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            nb += 1; tot += loss.item()
            if bi % 10 == 0 and bi:
                log(f"    batch {bi}/{n_batches}: loss {tot / nb:.4f}" + (f" (kd {kdt / nb:.4f})" if teacher is not None else "")
                    + f", {tok_seen / (time.time() - t_train):.0f} tokens/s (forward without gradients)")
        if nb == 0: break
        t_ep = time.time() - t0
        _, _, vm = evaluate(model, toks, cfg["section"], cfg)
        history.append(dict(epoch=ep + 1, train_loss=tot / nb, **{f"val_{k}": v for k, v in vm.items()},
                            train_seconds=t_ep, seconds=time.time() - t0, batches=nb))
        log(f"  epoch {ep + 1:2d}/{cfg['epochs']} ({nb} batches): train loss {tot / nb:.4f} | dev {fmt(vm)} | "
            f"train {t_ep:.0f}s, dev {time.time() - t0 - t_ep:.0f}s, weights moved {moved():.2e}")
        if vm["f1"] > best:
            best = vm["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
        if stop: break
    train_secs = time.time() - t_train
    model.load_state_dict(best_state, strict=False)
    best_ep = max(history, key=lambda h: h["val_f1"])["epoch"]

    # dev and test embeddings at the best epoch (+ other parts of the case), test submission
    emb, dev_sections = {}, {}
    for spec in [cfg["section"]] + list(cfg["eval_sections"]):
        t0 = time.time()
        Ed, Sd, vm_s = evaluate(model, toks, spec, cfg, "dev")
        Et, St_, _ = evaluate(model, toks, spec, cfg, "test")
        emb[spec] = dict(dev=dict(ids=split_cases("dev"), emb=Ed.half()), test=dict(ids=split_cases("test"), emb=Et.half()))
        dev_sections[spec] = vm_s
        if spec == cfg["section"]:
            vm, infer_secs, S_test, S_dev = vm_s, time.time() - t0, St_, Sd
        else:
            log(f"  dev reading {spec:10s}: {fmt(vm_s)} ({time.time() - t0:.0f}s, dev + test)")
    log(f"best epoch {best_ep}: dev {fmt(vm)}")
    torch.save(emb, os.path.join(out_dir, "emb.pt"))      # for fusion with BM25 / ensembles offline
    preds = rankings("test", S_test)
    assert len(preds) == len(SPLITS["test"]["queries"]) and all(len(set(v)) == 7 for v in preds.values())
    with open(os.path.join(out_dir, "prediction.json"), "w", encoding="utf-8") as f:
        json.dump(preds, f)
    with zipfile.ZipFile(os.path.join(out_dir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(out_dir, "prediction.json"), "prediction.json")
    json.dump(rankings("dev", S_dev), open(os.path.join(out_dir, "dev_predictions.json"), "w"))

    # teacher embeddings of the training cases, for a KD run of this session
    if any(r.get("teacher") == name for r in RUNS):
        t0 = time.time(); ids = split_cases("train")
        Etr = encode(model, tr_ch, ids, cfg).half()
        TEACHER_EMB[name] = {d: Etr[i] for i, d in enumerate(ids)}
        log(f"teacher embeddings for {len(ids)} training cases ({time.time() - t0:.0f}s)")

    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else 0.0
    res = dict(name=name, model=cfg["model"], method=cfg["method"], section=f"{cfg['section']}:{cfg['n_chunks']}",
               chunk_len=cfg["chunk_len"], attn=cfg["attn"] if info["kind"] != "gru" else "bigru",
               layers=f"{info['n_layers']}/{info['full_layers']}", total_params=n_total, trainable_params=n_train,
               best_epoch=best_ep, epochs_done=history[-1]["epoch"], zero_shot_f1=history[0]["val_f1"],
               **{f"val_{k}": v for k, v in vm.items()}, dev_sections=dev_sections, train_minutes=train_secs / 60,
               test_inference_seconds=infer_secs, peak_gpu_gb=peak, history=history, config=cfg, checkpoint=info["folder"])
    json.dump(res, open(os.path.join(out_dir, "val_metrics.json"), "w"), indent=1)
    if cfg["save_weights"]:
        torch.save(dict(trainable={k: v.to(torch.bfloat16) for k, v in best_state.items()}, config=cfg,
                        backbone_config=info["config"]), os.path.join(out_dir, "trainable_weights.pt"))
    log(f"train {train_secs / 60:.1f} min, peak GPU memory {peak:.2f} GB, outputs in {out_dir}")
    RESULTS.append(res)
    write_summary()
    del model, opt, backbone; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return res

In [ ]:
# ---- summary table + one zip to download (rewritten after every run, so a cut-off session keeps finished runs) ----
_sum_lock = threading.Lock()
def write_summary():
    with _sum_lock:
        lines = [f"{'run':16s} {'model':5s} {'method':7s} {'input':7s} {'layers':6s} {'total':>8s} {'trainable':>10s} "
                 f"{'ep':>3s} {'0-shot':>6s} {'F1@7':>6s} {'P@7':>6s} {'R@7':>6s} {'R@100':>6s} {'train min':>9s} "
                 f"{'infer s':>7s} {'peak GB':>7s}"]
        for r in RESULTS:
            lines.append(f"{r['name']:16s} {r['model']:5s} {r['method']:7s} {r['section']:7s} {r['layers']:6s} "
                         f"{r['total_params'] / 1e6:7.1f}M {r['trainable_params'] / 1e6:9.3f}M {r['best_epoch']:3d} "
                         f"{r['zero_shot_f1']:6.4f} {r['val_f1']:6.4f} {r['val_precision']:6.4f} {r['val_recall']:6.4f} "
                         f"{r['val_recall_at_100']:6.4f} {r['train_minutes']:9.1f} {r['test_inference_seconds']:7.0f} "
                         f"{r['peak_gpu_gb']:7.2f}")
        for r in RESULTS:
            if len(r["dev_sections"]) > 1:
                lines.append(f"  {r['name']} reading other parts: " + ", ".join(f"{k} {v['f1']:.4f}" for k, v in r["dev_sections"].items()))
        lines.append(f"BM25 (no training) dev Micro-F1@7 {f1_at_7('dev', BM25['dev'])['f1']:.4f}")
        open(os.path.join(WORK, "phase7_summary.txt"), "w").write("\n".join(lines) + "\n")
        json.dump(RESULTS, open(os.path.join(WORK, "phase7_results_summary.json"), "w"), indent=1, default=str)
        bundle = os.path.join(WORK, "phase7_results_bundle.zip")
        with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
            for f in glob.glob(os.path.join(WORK, "**", "*"), recursive=True):
                if f.endswith((".txt", ".json", ".zip", ".pt")) and not f.endswith("phase7_results_bundle.zip") \
                        and os.sep + "smoke" + os.sep not in f[len(WORK):]:
                    z.write(f, os.path.relpath(f, WORK))

In [ ]:
# ---- run everything: one worker per GPU, each takes the next run from the queue (2 x T4 -> 2 runs at a time) ----
# torch.cuda.set_device is per thread, and DEVICE = "cuda" means "this thread's GPU", so every run stays on its GPU.
# A KD run waits until its teacher (in this session) has finished.
SELECTED = [r for r in RUNS if not RUN_ONLY or r["name"] in RUN_ONLY]
for r in SELECTED:
    assert r.get("teacher") is None or r["teacher"] in {s["name"] for s in SELECTED}, f"{r['name']}: teacher not selected"
GPU_IDS = json.loads(os.environ.get("P7_GPU_IDS", "null")) or (list(range(torch.cuda.device_count())) if DEVICE.type == "cuda" else [None])
GPU_IDS = GPU_IDS[:max(1, len(SELECTED))]
_queue, _qlock, _finished = list(SELECTED), threading.Lock(), set()

def _worker(gpu):
    if gpu is not None: torch.cuda.set_device(gpu)
    while True:
        with _qlock:
            if not _queue: return
            ready = [r for r in _queue if r.get("teacher") is None or r["teacher"] in _finished]
            run = ready[0] if ready else None
            if run: _queue.remove(run)
        if run is None:
            time.sleep(20); continue
        log(f"starting {run['name']}")
        try:
            if run.get("teacher") is not None and run["teacher"] not in TEACHER_EMB:
                raise RuntimeError(f"teacher {run['teacher']} failed: no teacher embeddings")
            run_experiment(run)
        except Exception as e:                           # one failing run must not lose the others
            import traceback
            log(f"RUN {run['name']} FAILED: {e!r}\n{traceback.format_exc()}")
            gc.collect()
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
        finally:
            with _qlock: _finished.add(run["name"])

section(f"RUNNING {[r['name'] for r in SELECTED]} on {len(GPU_IDS)} worker(s): GPUs {GPU_IDS}")
_threads = [threading.Thread(target=_worker, args=(g,), name=f"gpu{g if g is not None else 'cpu'}-{i}") for i, g in enumerate(GPU_IDS)]
for t in _threads: t.start()
for t in _threads: t.join()

In [ ]:
section("SUMMARY")
write_summary()
log(open(os.path.join(WORK, "phase7_summary.txt")).read())
log(f"bundle: {os.path.join(WORK, 'phase7_results_bundle.zip')} "
    f"({os.path.getsize(os.path.join(WORK, 'phase7_results_bundle.zip')) / 1e6:.1f} MB)")